<a href="https://colab.research.google.com/github/XW-ABAP/AI-For-Beginners/blob/main/examples/04_text_sentiment_cp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

"""
Simple Text Sentiment Analysis
================================

This example shows how to analyze the sentiment (emotion) of text.
It's a simplified version that teaches NLP concepts without complex libraries.

What you'll learn:
- Text preprocessing (cleaning and preparing text)
- Feature extraction (converting words to numbers)
- Sentiment classification (positive vs negative)

Use case: Determine if a movie review is positive or negative.
"""

In [1]:
import re
from collections import Counter

In [3]:
class SimpleSentimentAnalyzer:
    """
    A basic sentiment analyzer that learns from labeled examples.

    工作原理：
    1. 学习哪些词更多出现在正面/负面文本中
    2. 为每个词计算一个 "情感分数"
    3. 用这些分数预测新文本的情感
    """

    def __init__(self):
        # 存储每个词的情感分数（正面词为正，负面词为负）
        self.word_scores = {}
        # 是否已训练
        self.is_trained = False

    def preprocess_text(self, text):
        """
        清洗并准备文本：
        1. 转小写
        2. 去标点
        3. 分词
        4. 去掉长度 <= 2 的短词
        """
        text = text.lower()
        text = re.sub(r'[^a-z\s]', '', text)
        words = text.split()
        words = [w for w in words if len(w) > 2]
        return words

    def train(self, training_data):
        """
        从带标签的样本中学习情感模式。
        training_data: [(text, 'positive'/'negative'), ...]
        """
        print("🎓 Training sentiment analyzer...")

        positive_words = Counter()
        negative_words = Counter()

        for text, sentiment in training_data:
            words = self.preprocess_text(text)
            if sentiment == 'positive':
                positive_words.update(words)
            else:
                negative_words.update(words)

        # 为每个词计算情感分数：>0 偏正面，<0 偏负面
        all_words = set(positive_words.keys()) | set(negative_words.keys())

        for word in all_words:
            pos_count = positive_words[word]
            neg_count = negative_words[word]
            total = pos_count + neg_count
            # 加 1 平滑，避免分母为 0
            self.word_scores[word] = (pos_count - neg_count) / (total + 1)

        self.is_trained = True

        # 展示学到的部分词
        print(f"✅ Learned sentiment for {len(self.word_scores)} words")
        print("\n📊 Most positive words:")
        sorted_words = sorted(self.word_scores.items(), key=lambda x: x[1], reverse=True)
        for word, score in sorted_words[:5]:
            print(f"   '{word}': {score:+.3f}")

        print("\n📊 Most negative words:")
        for word, score in sorted_words[-5:]:
            print(f"   '{word}': {score:+.3f}")

    def analyze(self, text):
        """
        预测新文本的情感。
        返回 (sentiment, confidence, score)
        """
        if not self.is_trained:
            raise Exception("Please train the analyzer first!")

        words = self.preprocess_text(text)

        total_score = 0
        word_count = 0

        for word in words:
            if word in self.word_scores:
                total_score += self.word_scores[word]
                word_count += 1

        avg_score = total_score / word_count if word_count > 0 else 0

        sentiment = "positive" if avg_score > 0 else "negative"
        confidence = min(abs(avg_score) * 100, 100)  # 转百分比

        return sentiment, confidence, avg_score

In [4]:
def create_training_data():
    """创建示例训练数据（电影评论 + 标签）。"""
    return [
        # 正面评论
        ("This movie was absolutely amazing and wonderful! I loved every minute.", "positive"),
        ("Brilliant performance! The acting was superb and the story captivating.", "positive"),
        ("Fantastic film! Highly recommend to everyone. Best movie of the year!", "positive"),
        ("Loved it! Great storytelling and beautiful cinematography.", "positive"),
        ("Excellent movie with outstanding performances. A must watch!", "positive"),
        ("Amazing! This film exceeded all my expectations. Truly remarkable.", "positive"),
        ("Wonderful experience! The plot was engaging and entertaining.", "positive"),
        ("Superb direction and acting! One of the best films I've seen.", "positive"),

        # 负面评论
        ("Terrible movie. Waste of time and money. Very disappointed.", "negative"),
        ("Awful film! Poor acting and boring story. Would not recommend.", "negative"),
        ("Horrible! The worst movie I have ever seen. Extremely disappointing.", "negative"),
        ("Bad movie with terrible plot. Boring and predictable.", "negative"),
        ("Disappointing film. Poor execution and weak performances.", "negative"),
        ("Worst movie ever! Horrible acting and stupid storyline.", "negative"),
        ("Terrible experience. Boring and poorly made. Don't waste your time.", "negative"),
        ("Awful! Poor quality and uninteresting. Complete waste of time.", "negative"),
    ]


training_data = create_training_data()
print(f"📊 Training data: {len(training_data)} movie reviews")

📊 Training data: 16 movie reviews


In [5]:
analyzer = SimpleSentimentAnalyzer()
analyzer.train(training_data)

🎓 Training sentiment analyzer...
✅ Learned sentiment for 78 words

📊 Most positive words:
   'was': +0.750
   'amazing': +0.667
   'wonderful': +0.667
   'this': +0.667
   'loved': +0.667

📊 Most negative words:
   'poor': -0.750
   'terrible': -0.750
   'boring': -0.750
   'waste': -0.750
   'time': -0.750


In [6]:
# 查看几个典型词的得分
for w in ["amazing", "wonderful", "fantastic", "terrible", "boring", "awful"]:
    print(f"{w:>10} -> {analyzer.word_scores.get(w)}")

   amazing -> 0.6666666666666666
 wonderful -> 0.6666666666666666
 fantastic -> 0.5
  terrible -> -0.75
    boring -> -0.75
     awful -> -0.6666666666666666


In [7]:
test_reviews = [
    "This movie was fantastic! I really enjoyed it.",
    "Boring and terrible. Not worth watching.",
    "Amazing cinematography and wonderful acting!",
    "The worst film I've seen this year. Awful.",
    "Pretty good movie with some great moments.",
    "Disappointing and poorly directed.",
]

print("🧪 Testing on new movie reviews:")
print("=" * 70)

for i, review in enumerate(test_reviews, 1):
    sentiment, confidence, score = analyzer.analyze(review)
    indicator = "😊" if sentiment == "positive" else "😞"
    print(f"\nReview {i}: \"{review}\"")
    print(f"  {indicator} Sentiment: {sentiment.upper()}")
    print(f"  📊 Confidence: {confidence:.1f}%")
    print(f"  📈 Score: {score:+.3f}")

🧪 Testing on new movie reviews:

Review 1: "This movie was fantastic! I really enjoyed it."
  😊 Sentiment: POSITIVE
  📊 Confidence: 44.8%
  📈 Score: +0.448

Review 2: "Boring and terrible. Not worth watching."
  😞 Sentiment: NEGATIVE
  📊 Confidence: 53.8%
  📈 Score: -0.538

Review 3: "Amazing cinematography and wonderful acting!"
  😊 Sentiment: POSITIVE
  📊 Confidence: 33.6%
  📈 Score: +0.336

Review 4: "The worst film I've seen this year. Awful."
  😊 Sentiment: POSITIVE
  📊 Confidence: 11.3%
  📈 Score: +0.113

Review 5: "Pretty good movie with some great moments."
  😊 Sentiment: POSITIVE
  📊 Confidence: 12.5%
  📈 Score: +0.125

Review 6: "Disappointing and poorly directed."
  😞 Sentiment: NEGATIVE
  📊 Confidence: 44.0%
  📈 Score: -0.440


In [8]:
user_input = "This movie was absolutely brilliant and I loved it!"

sentiment, confidence, score = analyzer.analyze(user_input)
indicator = "😊" if sentiment == "positive" else "😞"

print(f"Text: \"{user_input}\"")
print(f"{indicator} Sentiment: {sentiment.upper()}")
print(f"📊 Confidence: {confidence:.1f}%")
print(f"📈 Score: {score:+.3f}")

Text: "This movie was absolutely brilliant and I loved it!"
😊 Sentiment: POSITIVE
📊 Confidence: 40.1%
📈 Score: +0.401


In [9]:
import ipywidgets as widgets
from IPython.display import display

text_box = widgets.Text(
    placeholder="输入一句影评...",
    layout=widgets.Layout(width='600px')
)
button = widgets.Button(description="分析", button_style='primary')
output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output()
        if not text_box.value.strip():
            print("请输入内容")
            return
        s, c, sc = analyzer.analyze(text_box.value)
        indicator = "😊" if s == "positive" else "😞"
        print(f"{indicator} Sentiment: {s.upper()}")
        print(f"📊 Confidence: {c:.1f}%")
        print(f"📈 Score: {sc:+.3f}")

button.on_click(on_click)
display(text_box, button, output)

Text(value='', layout=Layout(width='600px'), placeholder='输入一句影评...')

Button(button_style='primary', description='分析', style=ButtonStyle())

Output()